<a href="https://colab.research.google.com/github/vitoriaferreirap/DeepLearning/blob/main/PoseEstimation%20_IC/Inferencia_Generalizacao_Analise/Teste1_Inferencia_superanimal_quadruped.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SuperAnimal no DeepLabCut PyTorch!
- Este notebook (github deeplabcut) demonstra como usar nossos modelos SuperAnimal no DeepLabCut 3.0! Leia mais em Ye et al. Nature Communications 2024 (https://www.nature.com/articles/s41467-024-48792-2) sobre os modelos SuperAnimal disponíveis e acompanhe abaixo!

- Vamos começar: instale a versão mais recente do DeepLabCut no COLAB:
Além disso, certifique-se de estar conectado a uma GPU: vá ao menu, clique em Runtime > Alterar Tipo de Runtime > selecione "GPU"

- Modelo "fundação" que já vem pré-treinado em mais de 45 espécies, incluindo cavalos. (superanimal_quadruped)

-  pode usá-lo sem precisar rotular nada. Ele já entende a anatomia básica de um cavalo e identifica as articulações automaticamente

- HrnetW3

## DeepLabCut ModelZoo (A Ferramenta Mãe)
O **ModelZoo** é como uma "biblioteca pública" de modelos prontos que o DeepLabCut oferece, o SuperAnimal-Quadruped mora aqui dentro.

- SuperAnimal-Quadruped: É o modelo "generalista". Ele foi treinado com milhares de fotos de cavalos, cachorros, elefantes. Ele é ótimo para começar porque já entende o que é um "corpo com quatro patas".

- Versatilidade: Entende cavalo, mas também entende se um cachorro passar atrás.
- Pontos (Labels):	Usa um padrão de pontos que serve para qualquer quadrúpede.
- Precisão	Muito bom para "zero-shot" (quando você não quer rotular nada).

- SuperAnimal é uma vantagem aqui, porque esses modelos foram treinados com uma técnica chamada Video Adaptation. Isso significa que eles são "treinados para entender movimento". Se você der apenas uma imagem, ele funciona bem, mas se você der um vídeo, ele consegue ser ainda mais preciso porque "entende" o contexto do frame anterior.

In [1]:
!unzip /content/dados.zip -d /content/

Archive:  /content/dados.zip
   creating: /content/dados/
   creating: /content/dados/A/
  inflating: /content/dados/A/1.mp4  
  inflating: /content/dados/A/10.mp4  
  inflating: /content/dados/A/11.mp4  
  inflating: /content/dados/A/12.mp4  
  inflating: /content/dados/A/13.mp4  
  inflating: /content/dados/A/14.mp4  
  inflating: /content/dados/A/15.mp4  
  inflating: /content/dados/A/16.mp4  
  inflating: /content/dados/A/17.mp4  
  inflating: /content/dados/A/18.mp4  
  inflating: /content/dados/A/19.mp4  
  inflating: /content/dados/A/2.mp4  
  inflating: /content/dados/A/20.mp4  
  inflating: /content/dados/A/3.mp4  
  inflating: /content/dados/A/4.mp4  
  inflating: /content/dados/A/5.mp4  
  inflating: /content/dados/A/6.mp4  
  inflating: /content/dados/A/7.mp4  
  inflating: /content/dados/A/8.mp4  
  inflating: /content/dados/A/9.mp4  
   creating: /content/dados/B/
  inflating: /content/dados/B/1.mp4  
  inflating: /content/dados/B/10.mp4  
  inflating: /content/dados/B/11

In [ ]:
!pip install --pre deeplabcut

In [1]:
# validar uso da GPU
import tensorflow as tf
device_name = tf.test.gpu_device_name()
if "GPU" not in device_name:
    print("GPU não encontrada. Verifique as configurações do notebook.")
else:
    print(f"Sucesso! Conectado à GPU: {device_name}")

Sucesso! Conectado à GPU: /device:GPU:0


In [2]:
# conferir versões
import deeplabcut
import torch

print(f"DeepLabCut versão: {deeplabcut.__version__}")
print(f"PyTorch versão: {torch.__version__}")
print(f"CUDA disponível no PyTorch: {torch.cuda.is_available()}")

Loading DLC 3.0.0rc13...
DLC loaded in light mode; you cannot use any GUI (labeling, relabeling and standalone GUI)
DeepLabCut versão: 3.0.0rc13
PyTorch versão: 2.10.0+cu128
CUDA disponível no PyTorch: True


- Importa as bibliotecas necessárias para o código funcionar, incluindo o DeepLabCut e as funções específicas para o SuperAnimal e análise de imagens/vídeos. É obrigatória para que o Google Colab entenda todos os comandos que executará depois.

In [3]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

import deeplabcut
import deeplabcut.utils.auxiliaryfunctions as auxiliaryfunctions
from deeplabcut.modelzoo import build_weight_init
from deeplabcut.modelzoo.utils import (
    create_conversion_table,
    read_conversion_table_from_csv,
)
from deeplabcut.modelzoo.video_inference import video_inference_superanimal
from deeplabcut.pose_estimation_pytorch.apis import (
    superanimal_analyze_images,
)
from deeplabcut.utils.pseudo_label import keypoint_matching

- Permite selecionar um arquivo do computador para carregar na memória temporária do Colab e define o caminho desse arquivo na variável video_path.




In [4]:
import glob
import os

# Definimos as pastas de vídeos (A até E)
pastas_videos = ['A', 'B', 'C', 'D', 'E']
# Dicionário para guardar os caminhos de cada pasta separadamente
videos_por_pasta = {}

print("--- Mapeando Arquivos ---")

for letra in pastas_videos:
    # Busca todos os .mp4 dentro da subpasta específica (ex: /content/dados/A/*.mp4)
    caminho_busca = f'/content/dados/{letra}/*.mp4'
    videos_por_pasta[letra] = sorted(glob.glob(caminho_busca))

    print(f"Pasta {letra}: {len(videos_por_pasta[letra])} vídeos encontrados.")

# Exemplo de como acessar a lista da pasta A
video_path_A = videos_por_pasta['A']

--- Mapeando Arquivos ---
Pasta A: 20 vídeos encontrados.
Pasta B: 20 vídeos encontrados.
Pasta C: 20 vídeos encontrados.
Pasta D: 9 vídeos encontrados.
Pasta E: 7 vídeos encontrados.

--- Pasta F (Imagens) ---
Foram encontradas 13 imagens para análise.


- Configura qual "cérebro" o modelo vai usar. Define o tipo de animal, a arquitetura da rede neural e o detector que encontra o bicho na imagem.


In [5]:
# @markdown ---
# @markdown SuperAnimal Configurations
superanimal_name = "superanimal_quadruped"  # @param ["superanimal_topviewmouse", "superanimal_quadruped"]
model_name = "hrnet_w32"  # @param ["hrnet_w32", "resnet_50"]
detector_name = (
    "fasterrcnn_resnet50_fpn_v2"  # @param ["fasterrcnn_resnet50_fpn_v2", "fasterrcnn_mobilenet_v3_large_fpn"]
)

# @markdown ---
# @markdown What is the maximum number of animals you expect to have in an image
max_individuals = 2  # @param {type:"slider", min:1, max:30, step:1}


- Função principal:  pega os vídeos e aplica o modelo SuperAnimal neles para identificar os pontos do corpo dos cavalos frame a frame. É esta célula que vai gerar o resultado final para análise.

- Processa a sequência de frames e mostra como o modelo se comporta com o cavalo em movimento. Se usar esta célula como imagem, terá apenas um "retrato" parado, o que é insuficiente para avaliar a generalização em situações reais de uso.

- video_adapt=True: usar o video_adapt=True é como deixar o modelo "estudar" o vídeo antes da prova. Isso mascara as falhas reais do modelo.

- video_adapt=False: usar o video_adapt=False permite que o modelo seja avaliado "zero-shot" (sem qualquer ajuste aos dados a serem analisados) isso é necessário para medir a falha de domínio (domain gap).


In [ ]:
import os

# Lista das pastas que você já tem
letras = ['A', 'B', 'C', 'D', 'E']
base_caminho = '/content/dados'

print("--- Criando Pastas de Resultado ---")

for letra in letras:
    # Cria o nome da nova pasta (ex: Resultado_A)
    nova_pasta = os.path.join(base_caminho, f'Resultado_{letra}')

    # Verifica se ela já existe, se não, cria
    if not os.path.exists(nova_pasta):
        os.makedirs(nova_pasta)
        print(f"Pasta criada: {nova_pasta}")
    else:
        print(f"Pasta já existente: {nova_pasta}")

print("\nEstrutura pronta para receber os resultados!")

In [ ]:
# Lista das pastas para o loop
pastas_videos = ['A', 'B', 'C', 'D', 'E']

# 1. LOOP PARA VÍDEOS (Pastas A até E)
for letra in pastas_videos:
    # Busca os vídeos da pasta atual usando o dicionário que criamos com glob
    video_path_atual = videos_por_pasta[letra]

    if video_path_atual:
        print(f"\n>>> Analisando VÍDEOS da Pasta {letra} ({len(video_path_atual)} arquivos)")

        _ = deeplabcut.video_inference_superanimal(
            videos=video_path_atual,
            superanimal_name=superanimal_name,
            model_name=model_name,
            detector_name=detector_name,
            video_adapt=False,
            max_individuals=max_individuals,
            dest_folder=f"/content/dados/Resultado_{letra}/", # Salva organizado por pasta
            batch_size=16,
        )

print("\n Processamento completo de todas as pastas!")

## Gráfico de Confiança por video (Likelihood)


In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

pastas_resultados = ['Resultado_A', 'Resultado_B', 'Resultado_C', 'Resultado_D', 'Resultado_E']
base_dados = '/content/dados'

print("--- Gerando Gráficos de Confiança por Vídeo ---")

for pasta in pastas_resultados:
    caminho_pasta = os.path.join(base_dados, pasta)
    arquivos = sorted(glob.glob(os.path.join(caminho_pasta, '*.h5')))

    if not arquivos:
        continue

    print(f"\n📈 Processando vídeos da {pasta}...")

    for arq in arquivos:
        try:
            df = pd.read_hdf(arq)
            # Seleciona as colunas de 'likelihood' (confiança)
            confianca_data = df.iloc[:, 2::3]

            media = confianca_data.mean(axis=1)
            min_conf = confianca_data.min(axis=1)
            max_conf = confianca_data.max(axis=1)

            plt.figure(figsize=(12, 4))
            plt.plot(media.values, color='#27ae60', label='Confiança Média', linewidth=1.5)
            plt.fill_between(range(len(media)), min_conf, max_conf, color='#27ae60', alpha=0.15, label='Variação entre Pontos')

            nome_video = os.path.basename(arq).replace('.h5', '').split('DLC')[0]
            plt.title(f"Vídeo {nome_video}: Estabilidade da Pose", fontsize=12)
            plt.ylabel("Likelihood (Confiança)")
            plt.xlabel("Frame")
            plt.ylim(0, 1.05)
            plt.axhline(y=0.6, color='red', linestyle='--', alpha=0.3, label='Threshold (0.6)')
            plt.legend(loc='lower right', fontsize=8)
            plt.grid(True, alpha=0.2)
            plt.tight_layout()

            # Salva o gráfico na pasta correspondente
            nome_grafico = f"confianca_{nome_video}.png"
            plt.savefig(os.path.join(caminho_pasta, nome_grafico))
            plt.close()

        except Exception as e:
            print(f"Erro ao processar o vídeo {os.path.basename(arq)}: {e}")

print("\n Todos os gráficos de confiança individuais foram salvos!")

--- Gerando Gráficos de Confiança por Vídeo ---

📈 Processando vídeos da Resultado_A...

📈 Processando vídeos da Resultado_B...

📈 Processando vídeos da Resultado_C...

📈 Processando vídeos da Resultado_D...

📈 Processando vídeos da Resultado_E...

 Todos os gráficos de confiança individuais foram salvos!


## Gráfico de Distribuição de Erros (Incerteza Acumulada)
- Esse gráfico usa a técnica de KDE (Kernel Density Estimate), que é a forma correta no PyTorch de visualizar onde o modelo está "errando" mais.
- Esta célula agrupa todos os dados de confiança da pasta para mostrar a frequência de erros (predições abaixo de 0.6). Isso serve para comparar a dificuldade entre os cenários (ex: Pasta A vs Pasta D).

In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
import numpy as np

pastas_resultados = ['Resultado_A', 'Resultado_B', 'Resultado_C', 'Resultado_D', 'Resultado_E']
base_dados = '/content/dados'

print("--- Gerando Histogramas de Distribuição de Erros ---")

for pasta in pastas_resultados:
    caminho_pasta = os.path.join(base_dados, pasta)
    arquivos = sorted(glob.glob(os.path.join(caminho_pasta, '*.h5')))

    confiancas_totais = []

    if not arquivos:
        continue

    for arq in arquivos:
        try:
            df = pd.read_hdf(arq)
            confianca_data = df.iloc[:, 2::3]
            confiancas_totais.extend(confianca_data.values.flatten())
        except:
            continue

    if confiancas_totais:
        plt.figure(figsize=(10, 6))
        # Histograma com curva de densidade (KDE)
        sns.histplot(confiancas_totais, bins=50, kde=True, color='#e67e22', edgecolor='none')

        plt.axvline(0.6, color='red', linestyle='--', linewidth=2, label='Threshold de Erro (0.6)')
        plt.title(f"Distribuição de Erros - {pasta}", fontsize=14)
        plt.xlabel("Nível de Confiança (Likelihood)")
        plt.ylabel("Frequência de Predições")
        plt.xlim(-0.05, 1.05)
        plt.legend()
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()

        # Salva o histograma na pasta de resultado
        plt.savefig(os.path.join(caminho_pasta, f"distribuicao_erros_{pasta}.png"))
        plt.close()

        # Resumo estatístico para o console
        conf_array = np.array(confiancas_totais)
        media_conf = np.mean(conf_array)
        taxa_erro = (conf_array < 0.6).sum() / len(conf_array) * 100
        print(f"📊 {pasta} -> Confiança Média: {media_conf:.4f} | Taxa de Incerteza: {taxa_erro:.2f}%")

print("\n Histogramas de erro concluídos e salvos!")

--- Gerando Histogramas de Distribuição de Erros ---
📊 Resultado_A -> Confiança Média: -0.1381 | Taxa de Incerteza: 70.01%
📊 Resultado_B -> Confiança Média: -0.2680 | Taxa de Incerteza: 76.38%
📊 Resultado_C -> Confiança Média: -0.4686 | Taxa de Incerteza: 80.52%
📊 Resultado_D -> Confiança Média: -0.3719 | Taxa de Incerteza: 79.21%
📊 Resultado_E -> Confiança Média: -0.2606 | Taxa de Incerteza: 75.04%

 Histogramas de erro concluídos e salvos!


In [19]:
from google.colab import drive
import os
import shutil

# Montar o Google Drive
drive.mount('/content/drive')

# Caminhos no Drive
drive_path = '/content/drive/MyDrive/Beckup_IC/FASE_2_IC/Aprendizado_Generalização'
nova_pasta_nome = 'Teste1_inferência_superanimal_quadruped'
caminho_final_drive = os.path.join(drive_path, nova_pasta_nome)

# Garantir que a estrutura de pastas existe no Drive
os.makedirs(caminho_final_drive, exist_ok=True)

# Origem dos dados no Colab
origem_local = '/content/dados'

print(f"Iniciando upload para: {caminho_final_drive}...")
print("Ignorando arquivos e pastas relacionados ao cenário F.")

# Lista de itens para ignorar
pastas_para_pular = ['F', 'Resultado_F']

for item in os.listdir(origem_local):
    # Se o item estiver na lista negra, a gente pula para o próximo
    if item in pastas_para_pular:
        continue

    s = os.path.join(origem_local, item)
    d = os.path.join(caminho_final_drive, item)

    try:
        if os.path.isdir(s):
            if os.path.exists(d):
                shutil.rmtree(d)
            shutil.copytree(s, d)
            print(f" ✅ Pasta {item} enviada.")
        else:
            # Copia arquivos soltos, se houver (como o vídeo temporário que criamos)
            if not item.endswith('.mp4') or 'F' not in item:
                shutil.copy2(s, d)
                print(f" ✅ Arquivo {item} enviado.")
    except Exception as e:
        print(f"Erro ao copiar {item}: {e}")

print("\n Backup concluído com sucesso (Pastas A a E)!")

Mounted at /content/drive
Iniciando upload para: /content/drive/MyDrive/Beckup_IC/FASE_2_IC/Aprendizado_Generalização/Teste1_inferência_superanimal_quadruped...
Ignorando arquivos e pastas relacionados ao cenário F.
 ✅ Pasta D enviada.
 ✅ Pasta Resultado_B enviada.
 ✅ Pasta B enviada.
 ✅ Pasta Resultado_C enviada.
 ✅ Pasta A enviada.
 ✅ Pasta E enviada.
 ✅ Pasta C enviada.
 ✅ Pasta Resultado_E enviada.
 ✅ Pasta Resultado_A enviada.
 ✅ Pasta Resultado_D enviada.

 Backup concluído com sucesso (Pastas A a E)!
